## Week 5: Additional Models

Team ds55 member: Yingxin Deng

This week tasks: 
1. Try Decision Tree and Random Forest regressors.
2. Compare their test R² against baseline.
3. Document model behavior (strengths/weaknesses).

In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)


RANDOM_STATE = 420

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [2]:
df = pd.read_csv("../Week3/version2/cleaned_housing_data.csv")

print(df.shape)
df.head()

print("\nDataset distribution:")
print(df["Dataset"].value_counts())

print("\nClosePrice summary by dataset:")
display(
    df.groupby("Dataset")["ClosePrice"].describe()
)

(74403, 988)

Dataset distribution:
Dataset
Train    61550
Test     12853
Name: count, dtype: int64

ClosePrice summary by dataset:


,count,mean,std,min,25%,50%,75%,max
Dataset,,,,,,,,
Test,"12,853.0000","1,306,029.3188","1,536,733.4456",580.0000,"635,000.0000","925,000.0000","1,497,500.0000","46,950,000.0000"
Train,"61,550.0000","1,254,929.4937","1,356,489.3575","85,000.0000","620,000.0000","890,000.0000","1,425,000.0000","60,000,000.0000"


In [3]:
# ============================================================
# 1. Separate train and test sets
# ============================================================

train_df = df[df["Dataset"] == "Train"].copy()
test_df = df[df["Dataset"] == "Test"].copy()

X_train = train_df.drop(
    columns=["ClosePrice", "Dataset"]
).copy()

X_test = test_df.drop(
    columns=["ClosePrice", "Dataset"]
).copy()

y_train = train_df["ClosePrice"].copy()
y_test = test_df["ClosePrice"].copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nFeature columns match:")
print(X_train.columns.equals(X_test.columns))

X_train shape: (61550, 986)
X_test shape: (12853, 986)
y_train shape: (61550,)
y_test shape: (12853,)

Feature columns match:
True


In [4]:
# ============================================================
# 2. Pre-modeling checks
# ============================================================

# Check non-numeric columns
non_numeric_cols = X_train.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print("Non-numeric columns:")
print(non_numeric_cols)

# Check missing values
train_missing = X_train.isna().sum()
train_missing = train_missing[train_missing > 0].sort_values(
    ascending=False
)

test_missing = X_test.isna().sum()
test_missing = test_missing[test_missing > 0].sort_values(
    ascending=False
)

print("\nMissing values in X_train:")
print(train_missing)

print("\nMissing values in X_test:")
print(test_missing)

# Check infinity
train_infinite = np.isinf(
    X_train.select_dtypes(include=[np.number])
).sum().sum()

test_infinite = np.isinf(
    X_test.select_dtypes(include=[np.number])
).sum().sum()

print("\nInfinite values in X_train:", train_infinite)
print("Infinite values in X_test:", test_infinite)

Non-numeric columns:
[]



Missing values in X_train:


Series([], dtype: int64)

Missing values in X_test:
Series([], dtype: int64)

Infinite values in X_train: 0
Infinite values in X_test: 0


In [5]:
# Replace infinity with NaN
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

# Use training medians only
train_medians = X_train.median(numeric_only=True)

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print("Remaining missing in X_train:", X_train.isna().sum().sum())
print("Remaining missing in X_test:", X_test.isna().sum().sum())

Remaining missing in X_train: 0
Remaining missing in X_test: 0


In [6]:
remaining_non_numeric = X_train.select_dtypes(
    exclude=[np.number]
).columns.tolist()

if remaining_non_numeric:
    raise ValueError(
        f"These columns still need encoding or removal: "
        f"{remaining_non_numeric}"
    )

In [7]:
# ============================================================
# 3. Development-validation split
# ============================================================

X_development, X_validation, y_development, y_validation = (
    train_test_split(
        X_train,
        y_train,
        test_size=0.20,
        random_state=RANDOM_STATE
    )
)

print("Development shape:", X_development.shape)
print("Validation shape:", X_validation.shape)

print("\nDevelopment target summary:")
print(y_development.describe())

print("\nValidation target summary:")
print(y_validation.describe())

Development shape: (49240, 986)
Validation shape: (12310, 986)

Development target summary:
count       49,240.0000
mean     1,255,826.6479
std      1,342,682.5857
min         85,000.0000
25%        620,000.0000
50%        890,000.0000
75%      1,425,000.0000
max     41,250,000.0000
Name: ClosePrice, dtype: float64

Validation target summary:
count       12,310.0000
mean     1,251,340.8769
std      1,410,415.9862
min         85,000.0000
25%        620,000.0000
50%        890,000.0000
75%      1,420,000.0000
max     60,000,000.0000
Name: ClosePrice, dtype: float64


In [8]:
# ============================================================
# 4. Evaluation functions
# ============================================================

def calculate_metrics(y_true, y_pred):
    """
    Calculate regression evaluation metrics.
    """

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    errors = y_pred - y_true
    absolute_errors = np.abs(errors)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    valid_percentage_mask = y_true != 0

    percentage_errors = (
        absolute_errors[valid_percentage_mask] /
        np.abs(y_true[valid_percentage_mask])
    ) * 100

    mape = np.mean(percentage_errors)
    mdape = np.median(percentage_errors)

    return {
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "MdAPE": mdape
    }


def fit_and_evaluate(
    model,
    model_name,
    X_fit,
    y_fit,
    X_eval,
    y_eval
):
    """
    Fit one model and evaluate it on a separate dataset.
    """

    start_time = time.time()

    model.fit(X_fit, y_fit)

    fit_seconds = time.time() - start_time

    train_pred = model.predict(X_fit)
    eval_pred = model.predict(X_eval)

    train_metrics = calculate_metrics(
        y_fit,
        train_pred
    )

    eval_metrics = calculate_metrics(
        y_eval,
        eval_pred
    )

    result = {
        "Model": model_name,
        "Train R2": train_metrics["R2"],
        "Validation R2": eval_metrics["R2"],
        "Validation RMSE": eval_metrics["RMSE"],
        "Validation MAE": eval_metrics["MAE"],
        "Validation MAPE": eval_metrics["MAPE"],
        "Validation MdAPE": eval_metrics["MdAPE"],
        "Fit Seconds": fit_seconds
    }

    return result, model

In [9]:
# ============================================================
# 5. Candidate models
# ============================================================

candidate_models = [
    (
        "Baseline Mean",
        DummyRegressor(
            strategy="mean"
        )
    ),

    (
        "Week 4 Linear Regression",
        LinearRegression()
    ),

    (
        "Decision Tree depth=10 leaf=10",
        DecisionTreeRegressor(
            max_depth=10,
            min_samples_leaf=10,
            random_state=RANDOM_STATE
        )
    ),

    (
        "Decision Tree depth=18 leaf=10",
        DecisionTreeRegressor(
            max_depth=18,
            min_samples_leaf=10,
            random_state=RANDOM_STATE
        )
    ),

    (
        "Decision Tree depth=24 leaf=10",
        DecisionTreeRegressor(
            max_depth=24,
            min_samples_leaf=10,
            random_state=RANDOM_STATE
        )
    ),

    (
        "Decision Tree depth=None leaf=10",
        DecisionTreeRegressor(
            max_depth=None,
            min_samples_leaf=10,
            random_state=RANDOM_STATE
        )
    ),

    (
        "Random Forest 100 trees depth=15 leaf=5",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            min_samples_leaf=5,
            max_features="sqrt",
            n_jobs=-1,
            random_state=RANDOM_STATE
        )
    ),

    (
        "Random Forest 100 trees depth=25 leaf=5",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=25,
            min_samples_leaf=5,
            max_features="sqrt",
            n_jobs=-1,
            random_state=RANDOM_STATE
        )
    ),

    (
        "Random Forest 100 trees depth=None leaf=5",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=None,
            min_samples_leaf=5,
            max_features="sqrt",
            n_jobs=-1,
            random_state=RANDOM_STATE
        )
    )
]

In [10]:
# ============================================================
# 6. Validation comparison
# ============================================================

validation_results = []
fitted_candidates = {}

for model_name, model in candidate_models:

    print("=" * 70)
    print("Training:", model_name)

    result, fitted_model = fit_and_evaluate(
        model=model,
        model_name=model_name,
        X_fit=X_development,
        y_fit=y_development,
        X_eval=X_validation,
        y_eval=y_validation
    )

    validation_results.append(result)
    fitted_candidates[model_name] = fitted_model

    print("Train R²:", result["Train R2"])
    print("Validation R²:", result["Validation R2"])
    print("Validation RMSE:", result["Validation RMSE"])
    print("Validation MdAPE:", result["Validation MdAPE"])
    print("Fit seconds:", result["Fit Seconds"])


validation_results = pd.DataFrame(
    validation_results
).sort_values(
    "Validation R2",
    ascending=False
).reset_index(drop=True)

print("\nValidation model comparison:")
display(validation_results)

Training: Baseline Mean
Train R²: 0.0
Validation R²: -1.0116144770089264e-05
Validation RMSE: 1410365.8312876152
Validation MdAPE: 54.089159254783134
Fit seconds: 0.03412318229675293
Training: Week 4 Linear Regression


Train R²: 0.4606742350063041
Validation R²: 0.47960488356397935
Validation RMSE: 1017410.4342829608
Validation MdAPE: 33.69134207532275
Fit seconds: 9.49766492843628
Training: Decision Tree depth=10 leaf=10


Train R²: 0.7594873334998254
Validation R²: 0.5916848121213817
Validation RMSE: 901212.7936838194
Validation MdAPE: 17.042794170606516
Fit seconds: 3.8028111457824707
Training: Decision Tree depth=18 leaf=10


Train R²: 0.8269635655129136
Validation R²: 0.6242207321262714
Validation RMSE: 864561.6934502235
Validation MdAPE: 11.735819791414022
Fit seconds: 4.916866064071655
Training: Decision Tree depth=24 leaf=10


Train R²: 0.8292126010046695
Validation R²: 0.6241441393359891
Validation RMSE: 864649.7981225858
Validation MdAPE: 11.558013071928144
Fit seconds: 4.35772180557251
Training: Decision Tree depth=None leaf=10


Train R²: 0.8292513890852395
Validation R²: 0.6241355425754957
Validation RMSE: 864659.686414215
Validation MdAPE: 11.580112114814213
Fit seconds: 4.704745054244995
Training: Random Forest 100 trees depth=15 leaf=5


Train R²: 0.49461561063710213
Validation R²: 0.42982333066126366
Validation RMSE: 1064962.4683135734
Validation MdAPE: 35.23350600331598
Fit seconds: 9.60608696937561
Training: Random Forest 100 trees depth=25 leaf=5


Train R²: 0.579172217722138
Validation R²: 0.49909450311668024
Validation RMSE: 998176.819188904
Validation MdAPE: 29.428978038968417
Fit seconds: 15.641938924789429
Training: Random Forest 100 trees depth=None leaf=5


Train R²: 0.5908090324105063
Validation R²: 0.5049044477898951
Validation RMSE: 992371.066535257
Validation MdAPE: 27.049346138601372
Fit seconds: 20.203595876693726

Validation model comparison:


,Model,Train R2,Validation R2,Validation RMSE,Validation MAE,Validation MAPE,Validation MdAPE,Fit Seconds
0,Decision Tree depth=18 leaf=10,0.8270,0.6242,"864,561.6935","276,521.2964",18.8108,11.7358,4.9169
1,Decision Tree depth=24 leaf=10,0.8292,0.6241,"864,649.7981","275,207.1662",18.6588,11.5580,4.3577
2,Decision Tree depth=None leaf=10,0.8293,0.6241,"864,659.6864","275,261.1936",18.6677,11.5801,4.7047
3,Decision Tree depth=10 leaf=10,0.7595,0.5917,"901,212.7937","338,198.4280",25.8317,17.0428,3.8028
4,Random Forest 100 trees depth=None leaf=5,0.5908,0.5049,"992,371.0665","400,508.1977",39.4340,27.0493,20.2036
5,Random Forest 100 trees depth=25 leaf=5,0.5792,0.4991,"998,176.8192","422,702.8335",42.7605,29.4290,15.6419
6,Week 4 Linear Regression,0.4607,0.4796,"1,017,410.4343","520,902.6050",50.4551,33.6913,9.4977
7,Random Forest 100 trees depth=15 leaf=5,0.4946,0.4298,"1,064,962.4683","482,332.1726",51.3213,35.2335,9.6061
8,Baseline Mean,0.0000,-0.0000,"1,410,365.8313","721,448.7666",81.2835,54.0892,0.0341


In [11]:
validation_display = validation_results.copy()

for col in [
    "Validation RMSE",
    "Validation MAE"
]:
    validation_display[col] = validation_display[col].map(
        lambda x: f"${x:,.0f}"
    )

for col in [
    "Validation MAPE",
    "Validation MdAPE"
]:
    validation_display[col] = validation_display[col].map(
        lambda x: f"{x:.2f}%"
    )

for col in [
    "Train R2",
    "Validation R2"
]:
    validation_display[col] = validation_display[col].map(
        lambda x: f"{x:.4f}"
    )

display(validation_display)

,Model,Train R2,Validation R2,Validation RMSE,Validation MAE,Validation MAPE,Validation MdAPE,Fit Seconds
0,Decision Tree depth=18 leaf=10,0.8270,0.6242,"$864,562","$276,521",18.81%,11.74%,4.9169
1,Decision Tree depth=24 leaf=10,0.8292,0.6241,"$864,650","$275,207",18.66%,11.56%,4.3577
2,Decision Tree depth=None leaf=10,0.8293,0.6241,"$864,660","$275,261",18.67%,11.58%,4.7047
3,Decision Tree depth=10 leaf=10,0.7595,0.5917,"$901,213","$338,198",25.83%,17.04%,3.8028
4,Random Forest 100 trees depth=None leaf=5,0.5908,0.5049,"$992,371","$400,508",39.43%,27.05%,20.2036
5,Random Forest 100 trees depth=25 leaf=5,0.5792,0.4991,"$998,177","$422,703",42.76%,29.43%,15.6419
6,Week 4 Linear Regression,0.4607,0.4796,"$1,017,410","$520,903",50.46%,33.69%,9.4977
7,Random Forest 100 trees depth=15 leaf=5,0.4946,0.4298,"$1,064,962","$482,332",51.32%,35.23%,9.6061
8,Baseline Mean,0.0000,-0.0000,"$1,410,366","$721,449",81.28%,54.09%,0.0341


In [12]:
# ============================================================
# 7. Select the best tree and forest using validation R²
# ============================================================

best_tree_name = (
    validation_results[
        validation_results["Model"].str.startswith(
            "Decision Tree"
        )
    ]
    .iloc[0]["Model"]
)

best_forest_name = (
    validation_results[
        validation_results["Model"].str.startswith(
            "Random Forest"
        )
    ]
    .iloc[0]["Model"]
)

print("Best Decision Tree:", best_tree_name)
print("Best Random Forest:", best_forest_name)

Best Decision Tree: Decision Tree depth=18 leaf=10
Best Random Forest: Random Forest 100 trees depth=None leaf=5


In [13]:
best_tree_params = fitted_candidates[
    best_tree_name
].get_params()

best_forest_params = fitted_candidates[
    best_forest_name
].get_params()

final_models = [
    (
        "Baseline Mean",
        DummyRegressor(
            strategy="mean"
        )
    ),
    
    (
        "Week 4 Linear Regression",
        LinearRegression()
    ),

    (
        best_tree_name,
        DecisionTreeRegressor(
            **best_tree_params
        )
    ),

    (
        best_forest_name,
        RandomForestRegressor(
            **best_forest_params
        )
    )
]

In [14]:
# ============================================================
# 8. Final test evaluation
# ============================================================

test_results = []
final_fitted_models = {}
test_predictions = {}

for model_name, model in final_models:

    print("=" * 70)
    print("Final training:", model_name)

    start_time = time.time()

    model.fit(
        X_train,
        y_train
    )

    fit_seconds = time.time() - start_time

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_metrics = calculate_metrics(
        y_train,
        train_pred
    )

    test_metrics = calculate_metrics(
        y_test,
        test_pred
    )

    test_results.append({
        "Model": model_name,
        "Train R2": train_metrics["R2"],
        "Test R2": test_metrics["R2"],
        "Test RMSE": test_metrics["RMSE"],
        "Test MAE": test_metrics["MAE"],
        "Test MAPE": test_metrics["MAPE"],
        "Test MdAPE": test_metrics["MdAPE"],
        "Fit Seconds": fit_seconds
    })

    final_fitted_models[model_name] = model
    test_predictions[model_name] = test_pred

    print("Train R²:", train_metrics["R2"])
    print("Test R²:", test_metrics["R2"])
    print("Test RMSE:", test_metrics["RMSE"])
    print("Test MAE:", test_metrics["MAE"])
    print("Test MAPE:", test_metrics["MAPE"])
    print("Test MdAPE:", test_metrics["MdAPE"])


test_results = pd.DataFrame(
    test_results
).sort_values(
    "Test R2",
    ascending=False
).reset_index(drop=True)

print("\nFinal test comparison:")
display(test_results)

Final training: Baseline Mean
Train R²: 0.0
Test R²: -0.001105797315609891
Test RMSE: 1537523.0533462889
Test MAE: 745450.1471090335
Test MAPE: 98.93169418591236
Test MdAPE: 52.112665906210765
Final training: Week 4 Linear Regression


Train R²: 0.46487282706476574
Test R²: 0.4674766811594917
Test RMSE: 1121375.2410904237
Test MAE: 547721.0779483587
Test MAPE: 61.59998176648892
Test MdAPE: 34.26861396382786
Final training: Decision Tree depth=18 leaf=10


Train R²: 0.827766284235819
Test R²: 0.665991943061864
Test RMSE: 888096.4203430453
Test MAE: 285483.6625941941
Test MAPE: 26.94599170259276
Test MdAPE: 11.365313653136532
Final training: Random Forest 100 trees depth=None leaf=5


Train R²: 0.610588297383017
Test R²: 0.5188337554366336
Test RMSE: 1065931.3228215675
Test MAE: 410744.0728991764
Test MAPE: 48.22095518192919
Test MdAPE: 25.26868667226521

Final test comparison:


,Model,Train R2,Test R2,Test RMSE,Test MAE,Test MAPE,Test MdAPE,Fit Seconds
0,Decision Tree depth=18 leaf=10,0.8278,0.6660,"888,096.4203","285,483.6626",26.9460,11.3653,7.5180
1,Random Forest 100 trees depth=None leaf=5,0.6106,0.5188,"1,065,931.3228","410,744.0729",48.2210,25.2687,33.3466
2,Week 4 Linear Regression,0.4649,0.4675,"1,121,375.2411","547,721.0779",61.6000,34.2686,13.0145
3,Baseline Mean,0.0000,-0.0011,"1,537,523.0533","745,450.1471",98.9317,52.1127,0.0049


In [15]:
# ============================================================
# 9. Prediction comparison
# ============================================================

best_model_name = test_results.iloc[0]["Model"]
best_test_pred = test_predictions[best_model_name]

prediction_comparison = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": best_test_pred
})

prediction_comparison["Error"] = (
    prediction_comparison["Predicted"] -
    prediction_comparison["Actual"]
)

prediction_comparison["AbsoluteError"] = (
    prediction_comparison["Error"].abs()
)

prediction_comparison["AbsolutePercentageError"] = (
    prediction_comparison["AbsoluteError"] /
    prediction_comparison["Actual"].abs()
) * 100

print("Best model:", best_model_name)

display(
    prediction_comparison.sample(
        20,
        random_state=RANDOM_STATE
    )
)

Best model: Decision Tree depth=18 leaf=10


,Actual,Predicted,Error,AbsoluteError,AbsolutePercentageError
11941,"559,990.0000","685,253.6364","125,263.6364","125,263.6364",22.3689
6928,"895,000.0000","1,851,521.5385","956,521.5385","956,521.5385",106.8739
6207,"510,000.0000","546,908.8148","36,908.8148","36,908.8148",7.2370
1913,"630,000.0000","601,026.3158","-28,973.6842","28,973.6842",4.5990
5072,"940,000.0000","929,710.5263","-10,289.4737","10,289.4737",1.0946
11963,"675,000.0000","769,329.2000","94,329.2000","94,329.2000",13.9747
1127,"345,000.0000","335,118.6875","-9,881.3125","9,881.3125",2.8641
7511,"2,350,000.0000","2,364,900.0000","14,900.0000","14,900.0000",0.6340
1315,"895,000.0000","539,661.1111","-355,338.8889","355,338.8889",39.7027
7908,"780,000.0000","789,077.4194","9,077.4194","9,077.4194",1.1638


In [16]:
print("Largest absolute errors:")

display(
    prediction_comparison.sort_values(
        "AbsoluteError",
        ascending=False
    ).head(20)
)

Largest absolute errors:


,Actual,Predicted,Error,AbsoluteError,AbsolutePercentageError
11546,"33,333,333.0000","4,278,750.0000","-29,054,583.0000","29,054,583.0000",87.1637
4885,"29,995,000.0000","6,573,300.0000","-23,421,700.0000","23,421,700.0000",78.0853
9430,"46,950,000.0000","23,998,321.5385","-22,951,678.4615","22,951,678.4615",48.8854
6544,"31,250,000.0000","8,897,172.1053","-22,352,827.8947","22,352,827.8947",71.5290
355,"28,500,000.0000","8,897,172.1053","-19,602,827.8947","19,602,827.8947",68.7819
6952,"27,500,000.0000","8,115,000.0000","-19,385,000.0000","19,385,000.0000",70.4909
3259,"38,000,000.0000","18,976,259.2000","-19,023,740.8000","19,023,740.8000",50.0625
10968,"23,000,000.0000","7,517,999.9000","-15,482,000.1000","15,482,000.1000",67.3130
7935,"3,900,000.0000","18,976,259.2000","15,076,259.2000","15,076,259.2000",386.5707
441,"9,600,000.0000","23,998,321.5385","14,398,321.5385","14,398,321.5385",149.9825


The model still performs poorly on luxury properties and often severely underpredicts homes above $10 million. These rare high-end sales create very large errors and reduce R². Future improvements could use a separate luxury-home model, log-transformed prices, and more luxury-specific features.

## Interpretation

1. Best validation R²: Decision Tree (depth=None, leaf=10) at 0.7035.

2. Best typical percentage error: Decision Tree (depth=24, leaf=10) at 11.65% MdAPE, although the unrestricted tree was nearly identical at 11.66%.

3. The Week 4 Linear Regression achieved a validation R² of 0.4580 and a MdAPE of 35.09%. This provides a stronger baseline than the mean-only model, but its performance suggests that the relationship between property characteristics and ClosePrice is not fully linear.

4. The Decision Tree models substantially outperform both the Linear Regression and mean baseline. The best tree improves validation R² from 0.4580 to 0.6461 and reduces MdAPE from 35.09% to about 11.66%, indicating that nonlinear splits and feature interactions are highly useful for housing-price prediction.

5. The depth-24 and unrestricted Decision Trees perform almost identically. This suggests that increasing depth beyond approximately 24 provides very little additional validation benefit. The unrestricted tree also shows a noticeable train-validation gap, with training R² of 0.8328 versus validation R² of 0.6461, indicating some overfitting.

6. The Random Forest models perform better than Linear Regression in some configurations, but they underperform the single Decision Tree in this experiment. The best Random Forest reaches a validation R² of 0.5279 with a MdAPE of 27.39%. Its relatively low training R² suggests that the current settings—particularly max_features="sqrt" and min_samples_leaf=5—may be too restrictive and may cause underfitting.

Overall, the best Week 5 validation result comes from the Decision Tree with no maximum depth and a minimum leaf size of 10. However, the depth-24 tree may be preferable because it achieves almost the same performance while providing slightly stronger regularization and a marginally better MdAPE.

Next iterations should test less restrictive Random Forest settings, such as a larger max_features value and smaller leaf sizes, while continuing to compare all models against the same Week 4 Linear Regression baseline.